In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Set the environment variable for Google credentials
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/home/karlchee/Keys/academic-era-488315-j5-52715453be94.json"

# Create engine without passing credentials in connect_args
engine = create_engine("bigquery://academic-era-488315-j5")

/tmp/ipykernel_1928/3779834692.py:9: SADeprecationWarning: The dbapi() classmethod on dialect classes has been renamed to import_dbapi().  Implement an import_dbapi() classmethod directly on class <class 'pybigquery.sqlalchemy_bigquery.BigQueryDialect'> to remove this warning; the old .dbapi() classmethod may be maintained for backwards compatibility.
  engine = create_engine("bigquery://academic-era-488315-j5")


In [5]:
# Extract the monthly revenue with month start date
query = """
WITH monthly_stamped_revenue AS (
    SELECT
        a.order_purchase_date,
        a.payment_value,
        b.year, b.month
    FROM olist_data_warehouse.order_items as a
    JOIN olist_data_warehouse.date as b
    ON a.order_purchase_date = b.full_date
)

SELECT
    DATE(CONCAT(CAST(year AS STRING), '-', LPAD(CAST(month AS STRING), 2, '0'), '-01')) as month_start,
    SUM(payment_value) as monthly_revenue
FROM monthly_stamped_revenue
GROUP BY month_start
ORDER BY month_start
"""

df_revenue = pd.read_sql(query, engine)
print(df_revenue.head(10))

  month_start  monthly_revenue
0  2016-09-01           354.75
1  2016-10-01         56808.84
2  2016-12-01            19.62
3  2017-01-01        137188.49
4  2017-02-01        286280.62
5  2017-03-01        432048.59
6  2017-04-01        412422.24
7  2017-05-01        586190.95
8  2017-06-01        502963.04
9  2017-07-01        584971.62


In [6]:
# Plot the revenue trend as a time series
import plotly.express as px
fig = px.line(df_revenue, x='month_start', y='monthly_revenue', markers=True,
              title='Monthly Revenue Trend Over Time')
fig.update_layout(xaxis_title='Month', yaxis_title='Revenue')
fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    "city": ["São Paulo", "Rio de Janeiro", "Belo Horizonte"],
    "lat": [-23.5505, -22.9068, -19.9167],
    "lon": [-46.6333, -43.1729, -43.9345],
    "revenue": [500000, 300000, 150000]
})

fig = px.scatter_geo(
    df,
    lat="lat",
    lon="lon",
    size="revenue",
    hover_name="city",
    color="revenue",
    color_continuous_scale="Reds",
    scope="south america",
    title="Sales Revenue by City in Brazil"
)
fig.show()

In [4]:

query = """
SELECT
  FORMAT_DATE('%Y-%m', oi.order_purchase_date) AS month,
  SUM(oi.payment_value) AS revenue,
  COUNT(DISTINCT oi.order_id) AS orders
FROM olist_data_warehouse.order_items oi
GROUP BY 1
ORDER BY 1;
"""

df_monthly = pd.read_sql(query, engine)
print(df_monthly.head())

# Plot the revenue trend as a time series
import plotly.express as px
fig = px.line(df_monthly, x='month', y='revenue', markers=True,
              title='Monthly Revenue Trend Over Time')
fig.update_layout(xaxis_title='Month', yaxis_title='Revenue')
fig.show()

     month    revenue  orders
0  2016-09     354.75       3
1  2016-10   56808.84     308
2  2016-12      19.62       1
3  2017-01  137188.49     789
4  2017-02  286280.62    1733
